In [1]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

19:23:27 [INFO] Device: cuda


In [ ]:
CFG = {
    "task":        "lo",
    "dataset":     "kcnh2",
    "fp_type":     "ecfp4",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {
        "train": train_df,
        "test": test_df,
    }

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} "
        f"n_train_clusters={train_df['cluster'].nunique()} | "
        f"test={len(test_df)} "
        f"n_test_clusters={test_df['cluster'].nunique()}"
    )

19:23:27 [INFO] Loaded lo/drd2 fold 1: train=2206, test=267
19:23:27 [INFO] Fold 1 | train=2206 y_mean=6.721 y_std=0.867 n_train_clusters=1 | test=267 n_test_clusters=50
19:23:27 [INFO] Loaded lo/drd2 fold 2: train=2128, test=267
19:23:27 [INFO] Fold 2 | train=2128 y_mean=6.736 y_std=0.840 n_train_clusters=1 | test=267 n_test_clusters=50
19:23:27 [INFO] Loaded lo/drd2 fold 3: train=2257, test=262
19:23:27 [INFO] Fold 3 | train=2257 y_mean=6.753 y_std=0.857 n_train_clusters=1 | test=262 n_test_clusters=50


In [5]:
folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

19:23:27 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/ecfp4_train_1.npz
19:23:27 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/ecfp4_test_1.npz
19:23:27 [INFO] Fold 1 | X_train: (2206, 1024), X_test: (267, 1024) | scale_features=False | y_mean=6.721 y_std=0.867
19:23:27 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/ecfp4_train_2.npz
19:23:27 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/ecfp4_test_2.npz
19:23:27 [INFO] Fold 2 | X_train: (2128, 1024), X_test: (267, 1024) | scale_features=False | y_mean=6.736 y_std=0.840
19:23:27 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/ecfp4_train_3.npz
19:23:27 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/ecfp4_test_3.npz
19:23:27 [INFO] Fold 3 | X_train: (2257, 1024), X_tes

In [ ]:
n_inner_trainings = N_ITER * CFG["inner_k"] * len(CFG["outer_folds"])
n_final_trainings = N_SEEDS * len(CFG["outer_folds"])

logger.info(
    f"Planned trainings: "
    f"{n_inner_trainings} inner-search runs + "
    f"{n_final_trainings} final retraining runs"
)

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(
    fold_results,
    title="MLP kcnh2 Lo — ecfp4",
)

19:23:27 [INFO] Planned trainings: 900 inner-search runs + 9 final retraining runs
19:23:27 [INFO] 
OUTER FOLD 1
19:23:31 [INFO]   [1/150] inner MSE=0.4310 (score=-0.4310) (4s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
19:23:35 [INFO]   [2/150] inner MSE=0.4586 (score=-0.4586) (4s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
19:23:38 [INFO]   [3/150] inner MSE=0.5226 (score=-0.5226) (3s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
19:23:40 [INFO]   [4/150] inner MSE=0.4462 (score=-0.4462) (2s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
19:23:41 [INFO]   [5/150] inner MSE=0.5014 (score=-0.5014) (1s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
19:23:48 [INFO]   [6/150] inner MSE=0.4514 (score=-0.4514) (7s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
19:23:52 [INFO]   [7/150] inner MSE=0.4392 (score=-0.4392) (4s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
19:23:57 [INFO]   [8/150] inner MSE=0.4643 (score=-0.4643) (5s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
19:24:02 [INFO]   [9/150] inner MSE=0.4479 (score=-0.4479) (5s) |


MLP DRD2 Lo — ecfp4
Fold 1: mean_spearman=0.3342
Fold 2: mean_spearman=0.2401
Fold 3: mean_spearman=0.2529

Aggregated metrics:
  mean_spearman_mean: 0.2757
  mean_spearman_std: 0.0417
  std_spearman_mean: 0.4581
  std_spearman_std: 0.0071
  mean_r2_mean: -0.383
  mean_r2_std: 0.0705
  mean_mae_mean: 0.724
  mean_mae_std: 0.008
  n_clusters_mean: 50.0
  n_clusters_std: 0.0

Best hyperparameters per fold:
Fold 1: L=2 N=512 r=0.7 act=leaky_relu dropout=0.2 bn=True init=kaiming lr=3e-03 wd=5e-03 bs=256
Fold 2: L=3 N=1024 r=0.5 act=relu dropout=0.0 bn=True init=kaiming lr=2e-03 wd=5e-03 bs=128
Fold 3: L=3 N=512 r=1.0 act=leaky_relu dropout=0.3 bn=True init=kaiming lr=2e-03 wd=1e-02 bs=256


{'mean_spearman_mean': np.float64(0.2757),
 'mean_spearman_std': np.float64(0.0417),
 'std_spearman_mean': np.float64(0.4581),
 'std_spearman_std': np.float64(0.0071),
 'mean_r2_mean': np.float64(-0.383),
 'mean_r2_std': np.float64(0.0705),
 'mean_mae_mean': np.float64(0.724),
 'mean_mae_std': np.float64(0.008),
 'n_clusters_mean': np.float64(50.0),
 'n_clusters_std': np.float64(0.0)}